Setup

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import random
import shutil
import yaml

Paths

In [2]:
PROJECT_DIR = Path(".")

DATA_DIR = PROJECT_DIR / "data"

IMAGE_DIR = DATA_DIR / "selected_photos"

LANDMARKS_DIR = DATA_DIR / "landmarks"

YOLO_DIR = DATA_DIR / "yolo_pose_180"

print("Images:", IMAGE_DIR)
print("Landmarks:", LANDMARKS_DIR)
print("YOLO:", YOLO_DIR)

Images: data\selected_photos
Landmarks: data\landmarks
YOLO: data\yolo_pose_180


XMLs

In [3]:
xml_paths = sorted(
    LANDMARKS_DIR.glob("landmarks_P*/annotations.xml")
)

for path in xml_paths:
    print(path)

data\landmarks\landmarks_P001_P030\annotations.xml
data\landmarks\landmarks_P031_P060\annotations.xml
data\landmarks\landmarks_P061_P090\annotations.xml
data\landmarks\landmarks_P091_P120\annotations.xml
data\landmarks\landmarks_P121_P150\annotations.xml
data\landmarks\landmarks_P151_P180\annotations.xml


In [4]:
landmarks = {}
iris_masks = {}

for xml_path in xml_paths:

    print("Reading:", xml_path)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    images = root.findall(".//image")

    for image in images:

        image_name = image.attrib["name"]

        landmarks[image_name] = {}
        iris_masks[image_name] = {}

        # -------------------------
        # Points
        # -------------------------

        for point in image.findall(".//points"):

            label = point.attrib["label"]

            points_string = point.attrib["points"]

            x, y = map(
                float,
                points_string.split(",")
            )

            landmarks[image_name][label] = (x, y)

        # -------------------------
        # Polygons
        # -------------------------

        for polygon in image.findall(".//polygon"):

            label = polygon.attrib["label"]

            points_string = polygon.attrib["points"]

            polygon_points = []

            for point in points_string.split(";"):

                x, y = map(
                    float,
                    point.split(",")
                )

                polygon_points.append((x, y))

            iris_masks[image_name][label] = polygon_points

Reading: data\landmarks\landmarks_P001_P030\annotations.xml
Reading: data\landmarks\landmarks_P031_P060\annotations.xml
Reading: data\landmarks\landmarks_P061_P090\annotations.xml
Reading: data\landmarks\landmarks_P091_P120\annotations.xml
Reading: data\landmarks\landmarks_P121_P150\annotations.xml
Reading: data\landmarks\landmarks_P151_P180\annotations.xml


In [5]:
from pathlib import Path

IMAGE_DIR = Path("data/selected_photos")

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG"
}

image_files = [
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix in image_extensions
]

print("Number of photos:", len(image_files))

Number of photos: 180


In [6]:
photo_names = {
    p.name
    for p in image_files
}

annotation_names = set(landmarks.keys())

missing_photos = annotation_names - photo_names
extra_photos = photo_names - annotation_names

print("Annotated images:", len(annotation_names))
print("Photos:", len(photo_names))

print("\nMissing photos:")
print(missing_photos)

print("\nExtra photos:")
print(list(extra_photos)[:10])

Annotated images: 180
Photos: 180

Missing photos:
set()

Extra photos:
[]


Order of key points

In [7]:
left_keypoints = [
    "left_eye_inner_corner",
    "left_eye_outer_corner",
    "left_white_point"
]

right_keypoints = [
    "right_eye_inner_corner",
    "right_eye_outer_corner",
    "right_white_point"
]

Creating bounding box

In [8]:
def create_eye_bbox(
    landmarks_dict,
    keypoint_labels,
    padding_ratio=0.5
):

    points = np.array(
        [
            landmarks_dict[label]
            for label in keypoint_labels
        ],
        dtype=np.float32
    )

    x_min = points[:, 0].min()
    x_max = points[:, 0].max()

    y_min = points[:, 1].min()
    y_max = points[:, 1].max()

    width = x_max - x_min
    height = y_max - y_min

    # اگر ارتفاع نقاط خیلی کم باشد،
    # حداقل ارتفاعی بر اساس عرض چشم در نظر می‌گیریم
    if height < width * 0.3:
        height = width * 0.3

    x_padding = width * padding_ratio
    y_padding = height * padding_ratio

    x_min -= x_padding
    x_max += x_padding

    y_min -= y_padding
    y_max += y_padding

    return [
        x_min,
        y_min,
        x_max,
        y_max
    ]

Turning bbox into YOLO

In [9]:
def bbox_to_yolo(
    bbox,
    img_width,
    img_height
):

    x1, y1, x2, y2 = bbox

    x1 = max(0, min(x1, img_width))
    x2 = max(0, min(x2, img_width))

    y1 = max(0, min(y1, img_height))
    y2 = max(0, min(y2, img_height))

    x_center = (x1 + x2) / 2
    y_center = (y1 + y2) / 2

    width = x2 - x1
    height = y2 - y1

    return [
        x_center / img_width,
        y_center / img_height,
        width / img_width,
        height / img_height
    ]

Creating YOLO label

In [10]:
def create_yolo_label(image_name):

    img_path = IMAGE_DIR / image_name

    img = cv2.imread(str(img_path))

    if img is None:
        raise FileNotFoundError(
            f"Could not load image: {img_path}"
        )

    img_height, img_width = img.shape[:2]

    image_landmarks = landmarks[image_name]

    lines = []

    # =================================
    # LEFT EYE
    # =================================

    left_bbox = create_eye_bbox(
        image_landmarks,
        left_keypoints,
        padding_ratio=0.5
    )

    left_bbox_yolo = bbox_to_yolo(
        left_bbox,
        img_width,
        img_height
    )

    left_points = [
        image_landmarks[label]
        for label in left_keypoints
    ]

    left_kps = []

    for x, y in left_points:

        left_kps.extend([
            x / img_width,
            y / img_height,
            2
        ])

    left_line = [
        0,
        *left_bbox_yolo,
        *left_kps
    ]

    lines.append(left_line)

    # =================================
    # RIGHT EYE
    # =================================

    right_bbox = create_eye_bbox(
        image_landmarks,
        right_keypoints,
        padding_ratio=0.5
    )

    right_bbox_yolo = bbox_to_yolo(
        right_bbox,
        img_width,
        img_height
    )

    right_points = [
        image_landmarks[label]
        for label in right_keypoints
    ]

    right_kps = []

    for x, y in right_points:

        right_kps.extend([
            x / img_width,
            y / img_height,
            2
        ])

    right_line = [
        0,
        *right_bbox_yolo,
        *right_kps
    ]

    lines.append(right_line)

    return lines

Split (train, validation, test)

In [11]:
image_names = sorted(
    annotation_names
)

random.seed(42)

random.shuffle(image_names)

train_names = image_names[:144]
val_names = image_names[144:162]
test_names = image_names[162:180]

print("Train:", len(train_names))
print("Val:", len(val_names))
print("Test:", len(test_names))

Train: 144
Val: 18
Test: 18


In [12]:
folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:

    folder_path = YOLO_DIR / folder

    folder_path.mkdir(
        parents=True,
        exist_ok=True
    )

print("YOLO folders created.")

YOLO folders created.


Creating the dataset

In [13]:
splits = {
    "train": train_names,
    "val": val_names,
    "test": test_names
}

for split, names in splits.items():

    print(f"\nProcessing {split}...")

    for image_name in names:

        # -------------------------
        # Copy image
        # -------------------------

        src = IMAGE_DIR / image_name

        dst = (
            YOLO_DIR
            / "images"
            / split
            / image_name
        )

        shutil.copy2(src, dst)

        # -------------------------
        # Create label
        # -------------------------

        lines = create_yolo_label(
            image_name
        )

        label_path = (
            YOLO_DIR
            / "labels"
            / split
            / f"{Path(image_name).stem}.txt"
        )

        with open(
            label_path,
            "w"
        ) as f:

            for line in lines:

                values = []

                for i, value in enumerate(line):

                    if i == 0:
                        values.append(
                            str(int(value))
                        )
                    else:
                        values.append(
                            str(float(value))
                        )

                f.write(
                    " ".join(values) + "\n"
                )

print("\nDataset created successfully!")


Processing train...

Processing val...

Processing test...

Dataset created successfully!


Creating data.yaml

In [14]:
yaml_content = """
path: data/yolo_pose_180

train: images/train
val: images/val
test: images/test

kpt_shape: [3, 3]

names:
  0: eye
"""

yaml_path = YOLO_DIR / "data.yaml"

with open(
    yaml_path,
    "w"
) as f:

    f.write(
        yaml_content.strip()
    )

print(
    "Created:",
    yaml_path
)

Created: data\yolo_pose_180\data.yaml


Training

In [15]:
from ultralytics import YOLO

model = YOLO(
    "yolo11n-pose.pt"
)

results = model.train(
    data=str(YOLO_DIR / "data.yaml"),

    epochs=100,

    imgsz=640,

    batch=4,

    patience=20,

    device="cpu",

    project="runs/eye_strabismus",

    name="yolo_pose_180",

    pretrained=True,

    workers=2,

    plots=True
)

Ultralytics 8.4.120  Python-3.14.6 torch-2.13.0+cpu CPU (12th Gen Intel Core i7-1255U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\yolo_pose_180\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_pose_180-2, nbs=6

In [17]:
from ultralytics import YOLO

MODEL_PATH = r"runs\pose\runs\eye_strabismus\yolo_pose_180-2\weights\best.pt"

model_180 = YOLO(MODEL_PATH)

print("Model loaded successfully!")
print(model_180)

Model loaded successfully!
YOLO(
  (model): PoseModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64

In [19]:
test_dir = YOLO_DIR / "images" / "test"

test_results = model_180.predict(
    source=str(test_dir),
    imgsz=640,
    conf=0.25,
    save=False,
    verbose=False
)

print("Number of predictions:", len(test_results))

Number of predictions: 18


In [20]:
from pathlib import Path

PREDICT_DIR = Path("runs/pose/predict_180")

test_results = model_180.predict(
    source=str(test_dir),
    imgsz=640,
    conf=0.25,
    save=True,
    project="runs/pose",
    name="predict_180",
    exist_ok=True,
    verbose=False
)

print("Number of predictions:", len(test_results))
print("Saved to:", PREDICT_DIR)

Results saved to C:\Users\Asus\Desktop\Files\EyeDerivationProject\runs\pose\runs\pose\predict_180
Number of predictions: 18
Saved to: runs\pose\predict_180


Metrics

In [26]:
import numpy as np
import pandas as pd
from pathlib import Path

all_errors_180 = []

label_map = {
    "left": [
        "left_eye_inner_corner",
        "left_eye_outer_corner",
        "left_white_point"
    ],
    "right": [
        "right_eye_inner_corner",
        "right_eye_outer_corner",
        "right_white_point"
    ]
}

for result in test_results:

    image_name = Path(result.path).name

    pred_points = result.keypoints.xy.cpu().numpy()
    pred_boxes = result.boxes.xyxy.cpu().numpy()

    # باید دقیقاً دو چشم پیدا شده باشد
    if len(pred_points) != 2:
        print(
            f"Warning: {image_name} -> "
            f"{len(pred_points)} objects detected"
        )
        continue

    # مرکز X هر Bounding Box
    box_centers_x = [
        (box[0] + box[2]) / 2
        for box in pred_boxes
    ]

    # سمت راست تصویر = چشم راست فرد
    # سمت چپ تصویر = چشم چپ فرد
    right_idx = np.argmin(box_centers_x)
    left_idx = np.argmax(box_centers_x)

    for eye_side, pred_idx in [
        ("left", left_idx),
        ("right", right_idx)
    ]:

        points = pred_points[pred_idx]

        for kp_idx, label in enumerate(label_map[eye_side]):

            pred_x, pred_y = points[kp_idx]
            gt_x, gt_y = landmarks[image_name][label]

            error = np.sqrt(
                (pred_x - gt_x)**2 +
                (pred_y - gt_y)**2
            )

            all_errors_180.append({
                "image": image_name,
                "eye": eye_side,
                "keypoint": label,
                "error_px": error
            })

errors_180 = pd.DataFrame(all_errors_180)

print("Evaluated keypoints:", len(errors_180))

Evaluated keypoints: 108


In [27]:
summary_180 = (
    errors_180
    .assign(
        point_type=errors_180["keypoint"].str.extract(
            r"(inner_corner|outer_corner|white_point)"
        )[0]
    )
    .groupby("point_type")["error_px"]
    .mean()
)

print(summary_180)

overall_180 = errors_180["error_px"].mean()

print("\nOverall Mean Error:", overall_180)

point_type
inner_corner    12.140041
outer_corner    14.448024
white_point      9.432309
Name: error_px, dtype: float32

Overall Mean Error: 12.006792
